# EDA — bruit.csv
Score de Vivabilité · Mesures acoustiques Paris

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

## 1. Chargement

In [ ]:
FILE = 'architecture-data/brute/score_de_vivabilité/bruit.csv'

df = pd.read_csv(FILE, sep=None, engine='python')
df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

## 2. Infos générales

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 3. Valeurs manquantes — critique pour ce dataset

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, max(3, len(df.columns) * 0.45)))
colors = ['#D85A30' if v > 50 else '#BA7517' if v > 20 else '#1D9E75' for v in missing.values]
ax.barh(missing.index, missing.values, color=colors)
ax.set_xlabel('% manquant')
ax.set_title('Valeurs manquantes par colonne (rouge > 50%, orange > 20%)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.axvline(50, color='#D85A30', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.show()

display(missing.rename('% manquant').to_frame())

## 4. Doublons

In [ ]:
n_dup = df.duplicated().sum()
print(f'Doublons : {n_dup} ({n_dup/len(df)*100:.2f}%)')
if n_dup > 0:
    display(df[df.duplicated(keep=False)].head(10))

## 5. Structure temporelle

In [ ]:
annee_col = [c for c in df.columns if 'ann' in c.lower() or 'year' in c.lower()][0]
print(f'Colonne année : {annee_col}')
print(f'Plage         : {df[annee_col].min()} → {df[annee_col].max()}')
print(f'Nb années     : {df[annee_col].nunique()}')
print(f'Années        : {sorted(df[annee_col].tolist())}')

## 6. Séparation colonnes Lden / Ln

In [ ]:
lden_cols = [c for c in df.columns if 'lden' in c.lower()]
ln_cols   = [c for c in df.columns if ('_ln_' in c.lower() or ' ln ' in c or c.lower().endswith('ln db(a)')) and c not in lden_cols]

print(f'Colonnes Lden : {len(lden_cols)}')
print(f'Colonnes Ln   : {len(ln_cols)}')

lden_ok = [c for c in lden_cols if df[c].isnull().mean() < 0.6]
ln_ok   = [c for c in ln_cols   if df[c].isnull().mean() < 0.6]
print(f'\nLden exploitables (< 60% NaN) : {len(lden_ok)}')
print(f'Ln   exploitables (< 60% NaN) : {len(ln_ok)}')

## 7. Évolution temporelle des capteurs Lden exploitables

In [ ]:
if lden_ok:
    fig, ax = plt.subplots(figsize=(13, 5))
    for col in lden_ok:
        serie = df[[annee_col, col]].dropna()
        label = col.replace(' - Lden dB(A)', '').replace(' - Lden  dB(A)', '').replace('lden_bruit routier_', '').strip()
        ax.plot(serie[annee_col], serie[col], marker='o', markersize=4, linewidth=1.5, label=label)
    ax.axhline(68, color='red', linestyle='--', linewidth=1, label='Seuil OMS Lden 68 dB')
    ax.set_title('Évolution Lden par capteur (dB(A))')
    ax.set_xlabel('Année'); ax.set_ylabel('Lden dB(A)')
    ax.legend(fontsize=7, loc='lower left', ncol=2)
    plt.tight_layout()
    plt.show()

## 8. Distribution des niveaux sonores (boxplots)

In [ ]:
all_ok = lden_ok + ln_ok
if all_ok:
    plot_data = df[all_ok].melt(var_name='Capteur', value_name='dB(A)').dropna()
    plot_data['Type'] = plot_data['Capteur'].apply(lambda x: 'Lden' if 'lden' in x.lower() else 'Ln')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, typ in zip(axes, ['Lden', 'Ln']):
        sub = plot_data[plot_data['Type'] == typ]
        if not sub.empty:
            sub.boxplot(column='dB(A)', by='Capteur', ax=ax)
            ax.set_title(f'Distribution {typ}')
            ax.set_ylabel('dB(A)'); ax.set_xlabel('')
            plt.sca(ax); plt.xticks(rotation=40, ha='right', fontsize=6)
    plt.suptitle('Niveaux sonores par capteur', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Couverture temporelle par capteur

In [ ]:
num_cols = df.select_dtypes(include='number').columns.drop(annee_col)
coverage = pd.DataFrame({
    'capteur'         : num_cols,
    'n_mesures'       : [df[c].notna().sum() for c in num_cols],
    '% couvert'       : [(df[c].notna().mean() * 100).round(1) for c in num_cols],
    'première_année'  : [df.loc[df[c].notna(), annee_col].min() if df[c].notna().any() else None for c in num_cols],
    'dernière_année'  : [df.loc[df[c].notna(), annee_col].max() if df[c].notna().any() else None for c in num_cols],
    'moy_dB'          : [df[c].mean().round(2) for c in num_cols],
}).sort_values('% couvert', ascending=False)
display(coverage)

## 10. Résumé + plan Silver

In [ ]:
print('=' * 55)
print('RÉSUMÉ EDA — bruit.csv')
print('=' * 55)
print(f'  Lignes (années)      : {len(df)}')
print(f'  Colonnes (capteurs)  : {len(df.columns) - 1}  (+1 col Année)')
print(f'  Plage temporelle     : {df[annee_col].min()} → {df[annee_col].max()}')
print(f'  Colonnes Lden        : {len(lden_cols)}')
print(f'  Colonnes Ln          : {len(ln_cols)}')
cols_trop_vides = (df.isnull().mean() > 0.6).sum()
print(f'  Capteurs > 60% NaN   : {cols_trop_vides}  ← à exclure en Silver')
print(f'  Capteurs exploitables: {len(all_ok)}')
print('=' * 55)
print()
print('ACTIONS SILVER REQUISES :')
print('  → Exclure capteurs avec > 60% de NaN')
print('  → Pivoter en format long : (année, capteur, type_Lden_Ln, valeur_dB)')
print('  → Normaliser les noms de capteurs (supprimer suffixes dB(A))')
print('  → Comparer Lden moyen annuel au seuil OMS 68 dB')
print('  → Export Parquet pour Gold scoring vivabilité')